# DQN (PER) training example on ImprovedB747Env

Этот ноутбук демонстрирует обучение DQN-агента (Prioritized Experience Replay) из `tensoraerospace/agent/dqn/model.py` на среде `ImprovedB747Env` из `tensoraerospace/envs/b747.py`.

Особенности:
- Дискретизация непрерывного действия среды до дискретного множества для совместимости с DQN
- Логирование в TensorBoard: метрики Loss/Q, TD-Error, PER/Beta, эпизодическая награда и epsilon
- Простая конфигурация среды и краткий запуск обучения


In [1]:
# Install optional display for TensorBoard (skip if already available)
# %pip install tensorboard --quiet

import os
import math
from typing import Any, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from gymnasium import spaces

from tensoraerospace.envs.b747 import ImprovedB747Env
from tensoraerospace.agent.dqn.model import Model as DQNModel, DQNAgent

# Select device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps")
print("Using device:", DEVICE)

# For reproducibility
np.random.seed(1)
torch.manual_seed(1)


Using device: mps


In [2]:
# Helper: Discretize continuous action space of ImprovedB747Env
# We wrap the env so that actions become discrete indices over a set of elevator deflections.

class DiscreteActionWrapper:
    def __init__(self, env: ImprovedB747Env, num_bins: int = 15):
        # symmetric discrete set in [-1, 1], inclusive
        self.env = env
        self.num_bins = int(num_bins)
        self.action_values = np.linspace(-1.0, 1.0, self.num_bins, dtype=np.float32)
        # Replace action_space to Discrete
        self.action_space = spaces.Discrete(self.num_bins)
        # Keep observation_space as original
        self.observation_space = env.observation_space

    def reset(self, *args, **kwargs):
        return self.env.reset(*args, **kwargs)

    def step(self, action_idx: int):
        action_norm = np.array([self.action_values[int(action_idx)]], dtype=np.float32)
        return self.env.step(action_norm)

    def render(self, *args, **kwargs):
        return self.env.render(*args, **kwargs)

    def close(self):
        return self.env.close()


In [3]:
# Build environment instance
from tensoraerospace.signals.standart import sinusoid_vertical_shift
from tensoraerospace.utils import convert_tp_to_sec_tp, generate_time_period

dt=0.1
num_steps = 3000
tp = generate_time_period(tn=20, dt=dt)
tps = convert_tp_to_sec_tp(tp, dt=dt)
number_time_steps = len(tp)

reference_signals = np.reshape(
    sinusoid_vertical_shift(
        tp=np.asarray(tps),
        frequency=0.05,
        amplitude=np.deg2rad(1.0),
        vertical_shift=0.0,
    ),
    [1, -1],
)
# initial state [u, w, q, theta] in SI (theta, q in rad)
init_state = np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)

base_env = ImprovedB747Env(
    initial_state=init_state,
    reference_signal=reference_signals,
    number_time_steps=num_steps,
    dt=0.1,
    initial_elevator_deg=0.0,
)

env = DiscreteActionWrapper(base_env, num_bins=11)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

# Quick reset to validate
obs, info = env.reset()
print("Initial obs shape:", np.array(obs).shape)


Observation space: Box(-1.0, 1.0, (4,), float32)
Action space: Discrete(11)
Initial obs shape: (4,)


In [4]:
# Build DQN model compatible with observation size and discrete actions

num_actions = env.action_space.n
obs_dim = int(np.prod(env.observation_space.shape))

# Our DQN Model is LazyLinear for the first layer; it infers input features
# from the first forward pass. We still construct it with action count.
model = DQNModel(num_actions=num_actions)
target_model = DQNModel(num_actions=num_actions)

print("num_actions:", num_actions)

# Quick forward to initialize lazy layer
_ = model(torch.zeros((1, obs_dim), dtype=torch.float32))


num_actions: 11


In [ ]:
# Configure PERAgent with TensorBoard logging

train_steps = 200000
agent = DQNAgent(
    model=model,
    target_model=target_model,
    env=env,
    learning_rate=1e-3,
    epsilon=0.2,
    epsilon_dacay=0.995,
    min_epsilon=0.05,
    gamma=0.99,
    batch_size=32,
    target_update_iter=1000,
    train_nums=train_steps,
    buffer_size=2000,
    replay_period=1,
    alpha=0.6,
    beta=0.4,
    beta_increment_per_sample=0.0005,
    log_dir=os.path.join("runs", "dqn_b747_improved"),
    verbose_histogram=False,
)
agent.device = DEVICE
model.to(DEVICE)
target_model.to(DEVICE)
print("Agent ready.")


Agent ready.


In [14]:
# Train
# Note: this runs a custom training loop inside PERAgent.
# View TensorBoard with:  %load_ext tensorboard ; %tensorboard --logdir runs

try:
    agent.train()
finally:
    agent.close()
    env.close()
print("Training finished.")


PERAgent Train:  37%|███▋      | 73444/199999 [10:55<18:49, 112.09step/s, loss=0.0040, eps=0.200] 


KeyboardInterrupt: 